# GAN으로 새로운 패션아이템 생성하기
*GAN을 이용하여 새로운 패션 아이템을 만들어봅니다*

이 프로젝트는 최윤제님의 파이토치 튜토리얼 사용 허락을 받아 참고했습니다.

* [yunjey/pytorch-tutorial](https://github.com/yunjey/pytorch-tutorial) - MIT License

In [1]:
import os # 운영체제 시스템 명령 사용을 위한 모듈
import torch # 파이토치 라이브러리
import torchvision # 이미지 처리 도구 모음
import torch.nn as nn # 신경망 구축을 위한 모듈
import torch.optim as optim # 최적화 알고리즘 모듈
from torchvision import transforms, datasets # 데이터셋 및 전처리 도구
from torchvision.utils import save_image # 이미지 저장 유틸리티
import matplotlib.pyplot as plt # 결과 시각화를 위한 도구
import numpy as np # 수치 계산을 위한 라이브러리

EPOCHS 과 BATCH_SIZE 등 학습에 필요한 하이퍼 파라미터 들을 설정해 줍니다.

In [ ]:
# 하이퍼파라미터 # 학습을 위한 주요 설정값
EPOCHS = 500 # 전체 데이터셋 학습 반복 횟수
BATCH_SIZE = 100 # 한 번에 처리할 이미지 묶음 크기
USE_CUDA = torch.cuda.is_available() # GPU 사용 가능 여부 확인
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 사용할 연산 장치 결정
print("Using Device:", DEVICE) # 현재 장치 출력

학습에 필요한 데이터셋을 로딩합니다. 

In [3]:
# Fashion MNIST 데이터셋 # 데이터 준비 섹션
trainset = datasets.FashionMNIST(
    './.data', # 데이터 저장 경로
    train=True, # 학습용 데이터 선택
    download=True, # 없으면 자동 다운로드
    transform=transforms.Compose([ # 전처리 과정 정의
       transforms.ToTensor(), # 이미지를 텐서로 변환
       transforms.Normalize((0.5,), (0.5,)) # -1 ~ 1 사이로 정규화
    ])
)
train_loader = torch.utils.data.DataLoader( # 배치 단위로 데이터를 불러오는 도구
    dataset     = trainset, # 대상 데이터셋
    batch_size  = BATCH_SIZE, # 배치 크기
    shuffle     = True # 무작위 섞기 설정
)

생성자는 64차원의 랜덤한 텐서를 입력받아 이에 행렬곱(Linear)과 활성화 함수(ReLU, Tanh) 연산을 실행합니다. 생성자의 결과값은 784차원, 즉 Fashion MNIST 속의 이미지와 같은 차원의 텐서입니다.

In [4]:
# 생성자 (Generator) # 가짜 이미지를 만드는 망
G = nn.Sequential(
        nn.Linear(64, 256), # 64차원 노이즈 입력을 256 노드로 확장
        nn.ReLU(), # 활성화 함수 ReLU
        nn.Linear(256, 256), # 256 노드에서 다시 256 노드로
        nn.ReLU(), # 활성화 함수 ReLU
        nn.Linear(256, 784), # 최종 출력인 784(28x28) 노드로
        nn.Tanh()) # 출력값을 -1 ~ 1 사이로 제한

판별자는 784차원의 텐서를 입력받습니다. 판별자 역시 입력된 데이터에 행렬곱과 활성화 함수를 실행시키지만, 생성자와 달리 판별자의 결과값은 입력받은 텐서가 진짜인지 구분하는 예측값입니다.

In [5]:
# 판별자 (Discriminator) # 진짜 이미지를 구별하는 망
D = nn.Sequential(
        nn.Linear(784, 256), # 784차원 이미지를 입력받아 256 노드로
        nn.LeakyReLU(0.2), # 음수 기울기가 0.2인 LeakyReLU 활성화
        nn.Linear(256, 256), # 256 노드에서 다시 256 노드로
        nn.LeakyReLU(0.2), # 활성화 함수
        nn.Linear(256, 1), # 최종적으로 1개의 출력값 생성
        nn.Sigmoid()) # 0(가짜) ~ 1(진짜) 사이의 확률값 출력

생성자와 판별자 학습에 쓰일 오차 함수와 최적화 알고리즘도 정의해 줍니다.

In [6]:
# 모델의 가중치를 지정한 장치로 보내기 # 모델 준비 섹션
D = D.to(DEVICE) # 판별자 장치 할당
G = G.to(DEVICE) # 생성자 장치 할당

# 이진 크로스 엔트로피 (Binary cross entropy) 오차 함수와 # 학습 설정
# 생성자와 판별자를 최적화할 Adam 모듈
criterion = nn.BCELoss() # 바이너리 크로스 엔트로피 손실 함수
d_optimizer = optim.Adam(D.parameters(), lr=0.0002) # 판별자용 Adam 최적화기
g_optimizer = optim.Adam(G.parameters(), lr=0.0002) # 생성자용 Adam 최적화기

모델 학습에 필요한 준비는 끝났습니다. 그럼 본격적으로 GAN을 학습시키는 loop을 만들어 보겠습니다. 

In [7]:
total_step = len(train_loader) # 전체 배치 단계 수
for epoch in range(EPOCHS): # 설정된 에폭만큼 반복 학습
    for i, (images, _) in enumerate(train_loader): # 배치 단위 이미지 순회
        images = images.reshape(BATCH_SIZE, -1).to(DEVICE) # 이미지를 평탄화하여 장치로 전송
        
        # '진짜'와 '가짜' 레이블 생성 # 학습 정답지 생성
        real_labels = torch.ones(BATCH_SIZE, 1).to(DEVICE) # 진짜는 1
        fake_labels = torch.zeros(BATCH_SIZE, 1).to(DEVICE) # 가짜는 0
        
        # 판별자가 진짜 이미지를 진짜로 인식하는 오차를 예산 # 판별자 진짜 학습
        outputs = D(images) # 진짜 이미지를 판별자에 입력
        d_loss_real = criterion(outputs, real_labels) # 정답(1)과의 오차 계산
        real_score = outputs # 결과 점수 저장
        
        # 무작위 텐서로 가짜 이미지 생성 # 가짜 데이터 생성
        z = torch.randn(BATCH_SIZE, 64).to(DEVICE) # 64차원 랜덤 노이즈 생성
        fake_images = G(z) # 생성자를 통해 이미지 생성
        
        # 판별자가 가짜 이미지를 가짜로 인식하는 오차를 계산 # 판별자 가짜 학습
        outputs = D(fake_images) # 가짜 이미지를 판별자에 입력
        d_loss_fake = criterion(outputs, fake_labels) # 정답(0)과의 오차 계산
        fake_score = outputs # 결과 점수 저장
        
        # 진짜와 가짜 이미지를 갖고 낸 오차를 더해서 판별자의 오차 계산 # 판별자 전체 손실
        d_loss = d_loss_real + d_loss_fake

        # 역전파 알고리즘으로 판별자 모델의 학습을 진행 # 판별자 가중치 업데이트
        d_optimizer.zero_grad() # 판별자 기울기 초기화
        g_optimizer.zero_grad() # 생성자 기울기 초기화
        d_loss.backward() # 손실 역전파
        d_optimizer.step() # 판별자 업데이트
        
        # 생성자가 판별자를 속였는지에 대한 오차를 계산 # 생성자 학습 준비
        fake_images = G(z) # 가짜 이미지 재생성
        outputs = D(fake_images) # 판별자에 가짜 입력
        g_loss = criterion(outputs, real_labels) # 가짜를 진짜(1)로 믿게 하는 손실 계산
        
        # 역전파 알고리즘으로 생성자 모델의 학습을 진행 # 생성자 가중치 업데이트
        d_optimizer.zero_grad() # 판별자 기울기 초기화
        g_optimizer.zero_grad() # 생성자 기울기 초기화
        g_loss.backward() # 손실 역전파
        g_optimizer.step() # 생성자 업데이트
        
    # 학습 진행 알아보기 # 각 에폭 종료 후 결과 출력
    print('Epoch [{}/{}], d_loss: {:.4f}, g_loss: {:.4f}, D(x): {:.2f}, D(G(z)): {:.2f}' 
          .format(epoch, EPOCHS, d_loss.item(), g_loss.item(), 
                  real_score.mean().item(), fake_score.mean().item()))

## 학습한 모델 저장하기

학습이 끝난 생성자(`G`)와 판별자(`D`)의 가중치를 파일로 저장합니다. `state_dict()`만 저장하는 것이 권장되는 방식입니다 (모델 구조는 코드에 있으므로).

In [8]:
# 저장 경로 준비 # 저장 설정
os.makedirs('./checkpoints', exist_ok=True) # 폴더 생성
G_PATH = './checkpoints/gan_generator.pth' # 생성자 가중치 경로
D_PATH = './checkpoints/gan_discriminator.pth' # 판별자 가중치 경로

# state_dict 저장 # 실제 파일 저장
torch.save(G.state_dict(), G_PATH) # 생성자 저장
torch.save(D.state_dict(), D_PATH) # 판별자 저장

print(f"생성자 저장: {G_PATH}") # 출력
print(f"판별자 저장: {D_PATH}") # 출력

학습이 끝난 생성자의 결과물을 한번 확인해 보겠습니다.

In [9]:
z = torch.randn(BATCH_SIZE, 64).to(DEVICE) # 테스트용 노이즈 생성
fake_images = G(z) # 생성자로 이미지 생성
for i in range(10): # 생성된 이미지 중 10개 시각화
    fake_images_img = np.reshape(fake_images.data.cpu().numpy()[i],(28, 28)) # 28x28 재구성
    plt.imshow(fake_images_img, cmap = 'gray') # 이미지 표시 설정
    plt.show() # 화면 출력

## 저장한 모델 불러와서 결과물 표시하기

학습된 가중치 파일을 다시 읽어와 새 생성자 모델 `G_loaded`에 적용한 뒤, 무작위 잠재 벡터로 이미지를 생성해 격자 형태로 표시합니다.

In [10]:
# 동일한 구조의 새 생성자 정의 # 로드 준비
G_loaded = nn.Sequential(
    nn.Linear(64, 256), # 구조 재정의
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 784),
    nn.Tanh()).to(DEVICE)

# 저장된 가중치 로드 # 데이터 불러오기
G_loaded.load_state_dict(torch.load(G_PATH, map_location=DEVICE)) # 가중치 적용
G_loaded.eval()  # 추론 모드로 전환
print(f"생성자 가중치 로드 완료: {G_PATH}")

# 잠재 벡터 → 이미지 생성 # 최종 테스트
N_SAMPLES = 16 # 샘플 수
with torch.no_grad(): # 기울기 계산 제외
    z = torch.randn(N_SAMPLES, 64).to(DEVICE) # 노이즈 생성
    fake_images = G_loaded(z).cpu().numpy() # 이미지 생성 및 변환

# 4x4 격자로 표시 # 시각화 마무리
fig, axes = plt.subplots(4, 4, figsize=(8, 8)) # 4x4 틀 생성
for i, ax in enumerate(axes.flatten()): # 각 칸에 이미지 배치
    img = fake_images[i].reshape(28, 28) # 28x28 형태
    ax.imshow(img, cmap='gray') # 표시
    ax.axis('off') # 축 끄기
plt.suptitle('Loaded Generator Output', fontsize=14) # 제목
plt.tight_layout() # 정렬
plt.show() # 출력